# Figure 2: cell-cloud metrics and PatchAE diagnostics

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Data preparation and model provenance

Panels a-b are metric schematics. Panels c-e use frozen outputs from the
K562 5,000-gene and cross-cell 3,352-gene PatchAE experiments. Training
entry points are registered as `K562-AE-PATHWAY` and
`XCL-AE-PATHWAY`; no training or model inference is run here.

In [ ]:
panel_map = REGISTRY.loc[REGISTRY["figure"].eq("Fig2")].copy()
required = ["source_data", "training_code", "evaluation_code", "plot_code", "canonical_panel"]
display(panel_map[["panel", "panel_type", "experiment_id", "claim_or_role", "status"]])

def archived_paths_exist(value, base):
    if value == "NA":
        return True
    return all((base / item).exists() for item in str(value).split(";"))

for column in ["source_data", "plot_code", "canonical_panel"]:
    missing = [
        value for value in panel_map[column]
        if not archived_paths_exist(value, ARCHIVE)
    ]
    assert not missing, f"Missing {column}: {missing}"
print("Panel-level figure inputs and plotting assets are present.")

In [ ]:
source = SOURCE_DATA / "Fig2"
display(pd.read_csv(source / "patchae_reconstruction_metrics.csv"))
display(pd.read_csv(source / "patchae_distribution_summary.csv"))
umap = pd.read_csv(source / "patchae_vs_globalvae_shared_umap.csv")
display(umap.groupby("source").size().rename("n_points"))

## Reproduce panels a-d from compact arrays

In [ ]:
subprocess.run(
    [sys.executable, str(ARCHIVE / "scripts" / "Fig2" / "make_figure2_vector_no_text.py")],
    check=True,
)

panels_a_to_d = [
    ("a", "Expression range coverage schematic", "figure2a_erc_labeled.svg"),
    ("b", "Correlation structure agreement schematic", "figure2b_csa_labeled.svg"),
    ("c", "Processed-expression marginal distributions", "figure2c_expression_distribution_labeled.svg"),
    ("d", "Clean latent marginal distributions", "figure2d_latent_distribution_labeled.svg"),
]
for panel, title, filename in panels_a_to_d:
    path = REPRO / "Fig2" / filename
    assert path.exists(), path
    display(Markdown(f"### Fig. 2{panel}: {title}"))
    display(SVG(filename=str(path)))

## Reproduce panel e from saved shared-reference UMAP coordinates

In [ ]:
prefix = REPRO / "Fig2" / "figure2_patchae_vae_shared_umap"
subprocess.run(
    [
        sys.executable,
        str(ARCHIVE / "scripts" / "Fig2" / "make_patchae_vae_umap_20260721.py"),
        "--coordinates-csv", str(source / "patchae_vs_globalvae_shared_umap.csv"),
        "--output-prefix", str(prefix),
    ],
    check=True,
)
display(Markdown("### Fig. 2e: PatchAE versus global-VAE shared-reference UMAP"))
display(SVG(filename=str(prefix.with_suffix(".svg"))))

In [ ]:
expected = [
    "figure2a_erc_labeled.svg",
    "figure2b_csa_labeled.svg",
    "figure2c_expression_distribution_labeled.svg",
    "figure2d_latent_distribution_labeled.svg",
    "figure2_patchae_vae_shared_umap.svg",
]
for name in expected:
    assert (REPRO / "Fig2" / name).exists(), name
print("All five labeled Fig. 2 subpanels were regenerated independently.")